# Standard MoE vs DeepSeek MoE: A Practical Comparison

So welcome to this notebook! In this notebook, we are gonna compare two different Mixture-of-Experts (MoE) architectures: the standard MoE approach used in many large language models versus the innovative DeepSeek-MoE architecture. We'll build both from scratch, train them on the same dataset, and see how they stack up against each other.

## What Are We Comparing?

**Standard MoE Architecture:**
- The "classic" approach used in models like Switch Transformers and GShard
- All experts are equal in size and function
- Uses an auxiliary loss to balance expert utilization
- Top-K routing sends each token to K experts

**DeepSeek MoE Architecture:**
- A newer approach with a hybrid expert system
- Combines "shared experts" (everyone uses them) with "routed experts" (only specific tokens use them)
- Uses a bias-only routing mechanism instead of auxiliary loss
- Aims for better quality and efficiency

## What Will We Learn?

By the end of this notebook, we'll understand:
1. How each architecture is implemented
2. Which one trains faster
3. Which one achieves better results
4. How experts are utilized in each approach

**Important Note:** The models we're building here are relatively small (only 6 layers with 512 embedding dimension) for demonstration purposes, so the differences won't be as dramatic as they would be at scale. In production-scale models with billions of parameters, these architectural differences would have much larger impacts on training efficiency and model quality.

In [12]:
!pip install datasets tiktoken tqdm numpy matplotlib pandas seaborn -q


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


## Setup and Environment Preparation

We'll start by installing the necessary dependencies for our experiment. This includes:
- `datasets`: To load and process the TinyStories dataset
- `tiktoken`: For tokenization using OpenAI's tokenizer (compatible with GPT-2)
- `tqdm`: For progress tracking during data processing and training
- `numpy`, `matplotlib`, `pandas`, `seaborn`: For data manipulation and visualization

These dependencies enable us to efficiently train our models and analyze the results in a systematic way.

In [13]:
# =============================================================================
# STAGE 1: SETUP, CONFIGURATION, AND DATA PREPARATION
# =============================================================================

# 1.1. Import Core Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import time
from contextlib import nullcontext
from dataclasses import dataclass

# 1.2. Import Helper Libraries
import numpy as np
from datasets import load_dataset
import tiktoken
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pandas as pd

print("All dependencies imported successfully.")

All dependencies imported successfully.


## Stage 1: Setup, Configuration, and Data Preparation

In this first stage, we'll:
1. Import necessary libraries for deep learning (PyTorch), data processing, and visualization
2. Configure the training parameters, including batch size, context length, learning rate, etc.
3. Define the DeepSeekMoEConfig class, which contains the architecture specifications for both models
4. Prepare and tokenize the TinyStories dataset

The DeepSeek MoE architecture has some key hyperparameters:
- `moe_n_routed_experts`: Number of specialized experts that process specific tokens (16 by default)
- `moe_top_k`: Number of experts selected per token (2 by default)
- `ds_moe_n_shared_experts`: Number of shared experts that process all tokens (2 by default)
- `ds_moe_shared_expert_hidden_dim`: Dimensionality of the shared experts (larger than routed experts)

For the standard MoE baseline, we'll adjust the number of experts to ensure both models have comparable parameter counts.

In [14]:
# 1.3. Global Training Configuration
BATCH_SIZE = 32
BLOCK_SIZE = 256
MAX_ITERS = 5000
EVAL_INTERVAL = 250
EVAL_ITERS = 100
LEARNING_RATE = 3e-4
WARMUP_ITERS = 200
MIN_LR = 3e-5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
PTDTYPE = {'float16': torch.float16, 'bfloat16': torch.bfloat16, 'float32': torch.float32}[DTYPE]
CTX = nullcontext() if DEVICE == 'cpu' else torch.amp.autocast(device_type=DEVICE, dtype=PTDTYPE)
SCALER = torch.cuda.amp.GradScaler(enabled=(DTYPE == 'float16'))

/tmp/ipykernel_279/1440405301.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  SCALER = torch.cuda.amp.GradScaler(enabled=(DTYPE == 'float16'))
